# Session 22 · LiteLLM — one call format, many models, fallbacks and cost
## BankAssist that doesn't go down, and knows what it costs

**Same format as Session 21:** you write the code in your own file, `s22/bankassist_llm.py`, in your own work
repository. This notebook explains every step in detail, and after each TODO a check cell runs **your** code.

| Part | What it is |
|---|---|
| **Part A · Read before you build** | Why one provider is not enough, what LiteLLM is, the words you need, how today works |
| **Part B · Build** | Section 1 sets up your file. Sections 2–6 are the TODOs. Section 7 breaks it on purpose. Sections 8–9 check and submit. |
| **Part C · Wrap-up** | Summary, honest limits, interview questions, homework |

**Where you type:** in `s22/bankassist_llm.py` in VS Code (Section 1 creates it). Run it from the course repo folder:
```
uv run python $env:WORK_REPO\s22\bankassist_llm.py       # Windows PowerShell
uv run python $WORK_REPO/s22/bankassist_llm.py             # macOS / Linux
```

---
# Part A · Read before you build

## A1. Why this session matters — three things that happen in real projects
Until now BankAssist talked to **one** model at **one** provider. Three things break that in real life:

1. **The provider has a bad day.** An outage, a slow response, or a `429 Too Many Requests` because the free tier
   ran out. Your customer sees an error page.
2. **You want a different model** — cheaper for simple questions, stronger for hard ones, or **on-premises** for
   customer data a bank may not send outside. With raw SDKs, each provider means new client code.
3. **Your manager asks, "What does one answer cost?"** Without per-request token and cost logging, nobody knows
   until the invoice arrives.

**LiteLLM** answers all three with one library: **one call format** for 100+ providers, a **Router** with retries
and **fallbacks**, and a **price list** to turn tokens into money.

**Analogy:** a universal travel adapter. Your laptop plug (your code) stays the same; the adapter fits the socket
in any country (any provider). And if one socket is dead, you plug into the next one.

| You learn to... | It saves you from... |
|---|---|
| Call any provider with `completion(model="provider/model", ...)` | Learning a new SDK per provider |
| Swap models by changing one string | Rewriting code when you change provider |
| Add a fallback with a Router | An outage or rate limit becoming a customer-facing error |
| Log tokens, cost and latency per request | Surprise bills, and no data to choose models with |

**Why interviewers care:** "How would you make your LLM app resilient and cost-aware?" is a standard
AI-engineering question. After today you can answer it with code you wrote.

## A2. Where this session fits
```
 S20  raw API: one provider, hand-written everything
 S21  LangChain: the PARTS (model object, prompts, @tool, your loop)
  │
 S22  LiteLLM: the LAYER UNDER your app ──► one call format · fallbacks · cost   ← today
  │
 S23  FastAPI + streaming: serve BankAssist over HTTP
 MP1  Mini-Project 1: memory + tools + LangChain + LiteLLM fallback + FastAPI + Docker
 M13  Cloud: the LiteLLM proxy as the company's LLM gateway (Azure OpenAI, Bedrock, Vertex behind one URL)
```
On the course's architecture slide, LiteLLM is **step 5, the "LLM gateway"** box: everything above it asks for a
model by name; LiteLLM decides which real provider answers, retries, falls back, and counts the cost.

## A3. The words you need today
Full table: `module03/terms_s22.md`. The six you'll use in the next hour:

- **Model string** — `provider/model`: `groq/openai/gpt-oss-20b`, `ollama_chat/llama3.2:3b`. Change the string, change the provider.
- **`completion()`** — LiteLLM's one function to call any chat model. The response looks exactly like OpenAI's.
- **Router** — knows several models by **group names you choose** (`bankassist`, `bankassist-backup`).
- **Fallback** — primary fails → Router tries the backup group. **Retry** — tries the *same* model again first.
- **Timeout** — how long to wait before a hung call counts as a failure.
- **Token cost** — `prompt_tokens × input price + completion_tokens × output price`, from LiteLLM's price list.

## A4. How today works

```
 course repo (pull, never edit)              YOUR work repo (write, commit, push)
 ┌──────────────────────────────┐            ┌────────────────────────────────────┐
 │ module03/s22_starter.py ─────┼── copy ───►│ s22/bankassist_llm.py  ← YOU TYPE  │
 │ module03/s22_check.py        │            └──────────────┬─────────────────────┘
 │ module03/s22_version_card.md │                           │ imported by the
 │ module03/s22_common_errors.md│◄──────────────────────────┘ check cells
 └──────────────────────────────┘
```
**The file already gives you four helpers — read them, don't change them:**
`params(model)` (key or local address for a model string), `messages_for(question)` (SYSTEM + question),
`answered_by(response)` (which model *really* answered), `fallbacks_used(response)` (how many fallbacks were tried).

**Your models** come from your `.env` — nothing new to set up:

| Your `.env` | PRIMARY | FALLBACK |
|---|---|---|
| Groq: `MODEL=openai/gpt-oss-20b` | `groq/openai/gpt-oss-20b` | `groq/llama-3.1-8b-instant` |
| Ollama: `MODEL=llama3.2:3b` | `ollama_chat/llama3.2:3b` | `groq/llama-3.1-8b-instant` (needs `GROQ_API_KEY=` in `.env`) |

Override either one with `LLM_PRIMARY=...` / `LLM_FALLBACK=...` in `.env`. If Groq has retired the fallback model,
run `uv run python list_models.py` and pick another small one.

**ChatGPT:** allowed — paste `module03/s22_version_card.md` first, **never paste your `.env` or key**, and ask it
to explain or fix *your* code.

---
# Part B · Build

## Section 1 · Set up your file
**What the next cell does:** finds the course repo root, loads `.env`, copies `s22_starter.py` to
`<WORK_REPO>/s22/bankassist_llm.py` **only if it isn't there yet** (re-running never wipes your work), and defines
`show(step)`, which runs the checker on your file and prints the lines for one TODO.

Before running it: `git pull` and `uv sync` in the course repo — one new package today, `litellm==1.80.0`.

In [ ]:
import os, sys, shutil
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "module02").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "module02"))
sys.path.insert(0, str(ROOT / "module03"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

WORK_REPO = Path(os.getenv("WORK_REPO") or r"C:\Users\you\Documents\genai-course-work")   # <-- your folder
target = WORK_REPO / "s22" / "bankassist_llm.py"
if target.exists():
    print("Already there - NOT overwritten (your work is safe):", target)
else:
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(ROOT / "module03" / "s22_starter.py", target)
    print("Created:", target)
print("\nNow open it in VS Code:   code", target)

from s22_check import check_s22_work, load_student
def show(step):
    """Run the checker on YOUR file and print only the lines for one TODO."""
    for r in check_s22_work(WORK_REPO):
        if step in r.name or (r.name.startswith(("work repo", "s22/", "your file")) and not r.passed):
            print(("PASS  " if r.passed else "FAIL  ") + r.name + (f"\n      -> {r.detail}" if r.detail else ""))

mod = load_student(WORK_REPO)
print("PRIMARY  =", mod.PRIMARY)
print("FALLBACK =", mod.FALLBACK)

**What you should see:** `Created: ...` (or `Already there`), and your two model strings.

**Check yourself:** run your file once, untouched — every step prints `not done yet (TODO n)`. That's the starting line.

## Section 2 · TODO 1 — one call, LiteLLM style
**What and why.** In S20 you called `client.chat.completions.create(...)` on an OpenAI client pointed at Groq. LiteLLM
replaces the client with **one function**, `completion()`, and the provider comes from the **model string**.

**The idea.**
```
 completion(model="groq/openai/gpt-oss-20b", messages=[...])
               └─┬─┘ └────────┬────────┘
             provider    the model at that provider
```
The response has the **same shape as OpenAI's**: `response.choices[0].message.content`, `response.usage.total_tokens`.
That's on purpose — everything you learned in S9–S20 still applies.

**What to type** — in `ask_once()`, replace the `raise NotImplementedError("TODO 1")` line with:
```python
return completion(**params(model), messages=messages_for(question), temperature=0, max_tokens=1500)
```
`**params(model)` unpacks the dict `{"model": ..., "api_key": ...}` into keyword arguments — the same as writing them out.

In [ ]:
show("TODO 1")
mod = load_student(WORK_REPO)
try:
    r = mod.ask_once(mod.PRIMARY, "When are branches open on Saturday?")
    print()
    print("text  :", r.choices[0].message.content)
    print("model :", r.model, "| tokens:", r.usage.prompt_tokens, "in +", r.usage.completion_tokens, "out")
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see:** `PASS TODO 1`, an answer mentioning 10am–1pm, and the token counts.

**Check yourself:** what would you change to ask Claude or Gemini instead? (Only the model string — and a key for
that provider. No new SDK, no new client.) *Common mistake:* a model string without a provider (`gpt-oss-20b`) →
`LLM Provider NOT provided`.

## Section 3 · TODO 2 — same code, two models
**What and why.** The real payoff of one call format: comparing models is a **loop over strings**. Teams do this to
choose the cheapest model that still passes their golden set (S17).

**What to type** — in `compare()`:
```python
out = []
for model in (PRIMARY, FALLBACK):
    response = ask_once(model, question)
    out.append((model, response.choices[0].message.content))
return out
```

In [ ]:
show("TODO 2")
mod = load_student(WORK_REPO)
try:
    for model, text in mod.compare("What is the home loan floating rate?"):
        print(f"\n{model}\n   {text}")
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see:** two answers, both saying 8.5% floating — from two different models, with identical code.

**Check yourself:** the smaller model is cheaper and faster. Would you trust it for EMI calculations? (Not without
measuring — that's what the golden set is for, and why S20 used a tool for the arithmetic anyway.)

## Section 4 · TODO 3 — a Router with a fallback
**What and why.** This is the heart of today. Your app should never ask for "Groq's gpt-oss-20b" directly. It asks
for a **role** — `bankassist` — and the Router decides who answers:

```
 your app ──► router.completion(model="bankassist")
                  │
                  ├─► "bankassist"        = PRIMARY   ── fails? retry once (num_retries=1)
                  │                                     still failing / timed out?
                  └─► "bankassist-backup" = FALLBACK  ── answers ✔   (fallbacks=[{"bankassist": ["bankassist-backup"]}])
```
**Retry vs fallback:** a retry knocks on the same door again (good for a one-second blip); a fallback goes next
door (good when the provider is down or rate-limiting you). **Timeout** decides how long you wait before giving up
on a door.

**What to type** — in `make_router()`:
```python
return Router(
    model_list=[
        {"model_name": "bankassist",        "litellm_params": params(primary)},
        {"model_name": "bankassist-backup", "litellm_params": params(fallback)},
    ],
    fallbacks=[{"bankassist": ["bankassist-backup"]}],
    num_retries=1,
    timeout=30,
)
```
The next cell builds a router whose primary **does not exist**, on purpose, and asks it a question.

In [ ]:
show("TODO 3")
mod = load_student(WORK_REPO)
import time
try:
    broken = mod.make_router(primary="groq/model-that-does-not-exist")
    t = time.perf_counter()
    r = broken.completion(model="bankassist", messages=mod.messages_for("When are branches open on Saturday?"))
    print()
    print("you asked for :", "bankassist  (primary = groq/model-that-does-not-exist)")
    print("answered by   :", mod.answered_by(r), f"after {mod.fallbacks_used(r)} fallback(s), {time.perf_counter() - t:.2f}s")
    print("answer        :", r.choices[0].message.content)
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see:** `PASS TODO 3`, then *answered by groq/llama-3.1-8b-instant after 1 fallback(s)*. The
customer got an answer even though the primary model doesn't exist. It took a little longer — the primary failed,
was retried once, then the fallback ran.

**Check yourself:** why does `answered_by()` matter? (Your logs, your cost, and your quality numbers must name the
model that **really** answered — not the one you asked for.)

## Section 5 · TODO 4 — what did that answer cost?
**What and why.** Every provider bills **per token**, with different prices for input (your prompt) and output
(the answer). LiteLLM ships a price list; `cost_per_token()` looks up the model and does the multiplication.

```
 cost = prompt_tokens × input price  +  completion_tokens × output price
 e.g. gpt-oss-20b on Groq (litellm 1.80.0 list price): 1,000 in + 500 out  =  $0.000100 + $0.000250  =  $0.000350
```
Local models (Ollama) aren't in the price list — they cost **$0 per token** (you pay for the laptop or GPU instead),
so your code must not crash on them.

**What to type** — in `cost_line()`:
```python
p, c = response.usage.prompt_tokens, response.usage.completion_tokens
try:
    cost_in, cost_out = cost_per_token(model=model, prompt_tokens=p, completion_tokens=c)
    cost = cost_in + cost_out
except Exception:
    cost = 0.0
return f"{model} | {p + c} tokens | ${cost:.6f} | {seconds:.2f}s"
```

In [ ]:
show("TODO 4")
mod = load_student(WORK_REPO)
try:
    t = time.perf_counter()
    r = mod.ask_once(mod.PRIMARY, "What is the home loan fixed rate?")
    print()
    print(mod.cost_line(r, mod.PRIMARY, time.perf_counter() - t))
    one = mod.cost_line(r, mod.PRIMARY, 0).split("$")[1].split(" ")[0]
    print(f"\n10,000 answers a day at this size ≈ ${float(one) * 10_000:.2f} a day")
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see:** one line like `groq/openai/gpt-oss-20b | 160 tokens | $0.000035 | 0.62s`, then a daily
estimate. One answer is a fraction of a cent; a bank with a million customers is a budget line.

**Check yourself:** these are **list prices** from the library's price file, not your invoice — free tiers, discounts
and currency are separate. Why still log them? (To compare models and catch a prompt that suddenly uses 10× the tokens.)

## Section 6 · TODO 5 (stretch) — every answer logged
Write `ask_logged(router, question)` below the `# TODO 5` comment in your file — the comment lists the four steps.
It answers **through the Router** and prints one cost line for the model that **really** answered. This is the
function Mini-Project 1 uses.

In [ ]:
mod = load_student(WORK_REPO)
if not hasattr(mod, "ask_logged"):
    print("TODO 5 not done yet (optional)")
else:
    router = mod.make_router()
    for q in ["When are branches open on Saturday?", "What is the home loan floating rate?"]:
        print(mod.ask_logged(router, q), "\n")

## Section 7 · Break it on purpose: what a fallback can and can't save
**What and why.** Fallbacks feel like magic. Here are two failures — predict each one before running.

**What the next cell does:** (1) a primary that doesn't exist → the fallback saves you; (2) **both** models broken →
no fallback can help. (A *slow* primary behaves like case 1 once `timeout` runs out — try `timeout=1` at home.)

In [ ]:
mod = load_student(WORK_REPO)
q = mod.messages_for("When are branches open on Saturday?")
cases = [
    ("primary missing", dict(primary="groq/model-that-does-not-exist")),
    ("both missing", dict(primary="groq/model-that-does-not-exist", fallback="groq/also-missing")),
]
for label, kw in cases:
    t = time.perf_counter()
    try:
        r = mod.make_router(**kw).completion(model="bankassist", messages=q)
        print(f"{label:16} -> answered by {mod.answered_by(r)} in {time.perf_counter() - t:.1f}s")
    except NotImplementedError as e:
        print("not done yet:", e); break
    except Exception as e:
        print(f"{label:16} -> FAILED after {time.perf_counter() - t:.1f}s: {type(e).__name__}")

**What you should see:**

| Case | Result | Lesson |
|---|---|---|
| primary missing | the backup answers, a little slower | fallbacks turn an outage into a delay |
| both missing | an error, after trying both | fallbacks don't help when **every** option is down — your app still needs a polite error message (S23) |

**Discuss:** if Groq itself is down, does a Groq fallback help? (No — real fallbacks go to a **different provider**:
Groq → OpenAI, or cloud → on-prem. Same provider = same outage.)

## Section 8 · Checkpoint
Offline — LiteLLM's test mode scripts the answers, so this uses no tokens.

In [ ]:
from prompt_utils import results_table
print(results_table(check_s22_work(WORK_REPO)))

## Section 9 · Commit and push
```
cd $env:WORK_REPO
git add s22/bankassist_llm.py
git commit -m "Session 22: BankAssist with LiteLLM - fallback and cost"
git push
cd <your course repo folder>
uv run pytest tests/test_s22.py -rs
```

---
# Part C · Wrap-up

## C1. Summary
- **LiteLLM** gives one `completion()` call for many providers; the **model string** picks the provider.
- A **Router** asks for a group name (`bankassist`) and falls back to another group when the primary fails.
- **Retry** = same model again; **fallback** = a different model; **timeout** = when to stop waiting.
- **Log cost per request** with the model that *really* answered; local models cost $0 per token.
- With LangChain (S21) you got the *parts*; with LiteLLM you get the *layer under* them — the LLM gateway.

## C2. Honest limits
- **Another dependency** that changes often — pinned at `litellm==1.80.0` for this course.
- **Fallback to the same provider** doesn't survive that provider's outage.
- **Different models answer differently** — a fallback can pass the call but fail your golden set. Test both.
- **List prices aren't your bill**; and a self-hosted proxy is one more server that can go down (run two).
- **Data residency:** a fallback to an outside provider may send data the primary kept in-house. Decide per data type.

## C3. Interview questions, answered
1. **How do you make an LLM app resilient to provider outages?** Timeouts, retries for blips, fallbacks to a
   *different* provider for outages, and a polite error when everything fails — all logged with the model that answered.
2. **Retry or fallback — which first?** Retry briefly for transient errors (one retry, backoff); fall back for
   rate limits, timeouts and hard failures. Never retry forever.
3. **How do you track LLM cost?** Tokens in/out per request × per-model price, logged with model, user and feature,
   aggregated in a dashboard with alerts — LiteLLM's price map or the proxy's spend tracking does the lookup.
4. **Why put a gateway (LiteLLM proxy) in front of models?** One place for keys, budgets per team, routing,
   fallbacks, logging and data-residency rules; apps just use one OpenAI-compatible URL.
5. **LangChain vs LiteLLM?** Different layers: LangChain builds the app (prompts, tools, chains, agents); LiteLLM
   standardises and routes the model calls underneath. Many teams use both.

## C4. Homework (choose at least two)
1. **TODO 5**, if you skipped it.
2. **Cheapest model that passes:** run S20's ten golden questions (`from tool_tasks import GOLDEN`) through
   `ask_once` on two models; report accuracy, tokens and cost for each. Which would you ship?
3. **Cross-provider fallback:** if you have Ollama, make the fallback `ollama_chat/llama3.2:3b` and pull your Wi-Fi
   during a call. What happens, and how long does it take?
4. **Budget guard:** keep a running total in `ask_logged`; refuse (politely) when a session passes $0.01.